# 00 - Data acquisition and integrity verification (can-train-and-test)
[Kaggle version] Downloads the DTU Data release (DOI 10.11583/DTU.24805533.v1, CC BY 4.0) inside Colab, checks the archive MD5 against the DTU record, and checks all 236 CSVs against the Stage 1B manifest via a digest over sorted `path,size,sha256` lines. No features, no training.

In [ ]:
import os, sys, time, json, platform, subprocess, shutil, hashlib
env = {'python': sys.version.split()[0]}
r = subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader', shell=True, capture_output=True, text=True)
env['gpu'] = r.stdout.strip() or 'no GPU'
env['ram_GB'] = round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30, 1)
env['cpus'] = os.cpu_count()
env['disk_free_GB'] = round(shutil.disk_usage('/tmp').free / 2**30, 1)
import torch
env['torch'] = torch.__version__
env['cuda'] = torch.cuda.is_available()
print(json.dumps(env))

In [ ]:
URL = 'https://ndownloader.figshare.com/files/43632393'
EXP_MD5 = 'bd6509d670c0a0009cb3ecab34111bcd'
EXP_SIZE = 1507455719
ZIP = '/tmp/can-train-and-test.zip'
t0 = time.time()
if not (os.path.exists(ZIP) and os.path.getsize(ZIP) == EXP_SIZE):
    subprocess.run(['wget', '-q', '-O', ZIP, URL], check=True)
dl_s = round(time.time() - t0, 1)
h = hashlib.md5()
with open(ZIP, 'rb') as f:
    for b in iter(lambda: f.read(8 << 20), b''):
        h.update(b)
zip_ok = os.path.getsize(ZIP) == EXP_SIZE and h.hexdigest() == EXP_MD5
print('download_s', dl_s, 'md5', h.hexdigest(), 'ZIP_MATCH' if zip_ok else 'ZIP_MISMATCH')

In [ ]:
DATA = '/tmp/data'
ROOT = DATA + '/can-train-and-test'
t0 = time.time()
if not os.path.isdir(ROOT):
    subprocess.run(['unzip', '-q', '-o', ZIP, '-d', DATA], check=True)
unzip_s = round(time.time() - t0, 1)
from concurrent.futures import ThreadPoolExecutor
paths = sorted(os.path.relpath(os.path.join(dp, f), ROOT).replace(os.sep, '/') for dp, _, fs in os.walk(ROOT) for f in fs if f.endswith('.csv'))
def sha(rel):
    p = ROOT + '/' + rel
    hh = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(8 << 20), b''):
            hh.update(b)
    return rel + ',' + str(os.path.getsize(p)) + ',' + hh.hexdigest()
t0 = time.time()
with ThreadPoolExecutor(4) as ex:
    lines = sorted(ex.map(sha, paths))
hash_s = round(time.time() - t0, 1)
EXP_N, EXP_BYTES = 236, 7471443340
EXP_DIGEST = '15111c4aad65c438c0cce8da77ff904944ab822ac06f3c91163bdd4e5680064b'
digest = hashlib.sha256('\n'.join(lines).encode()).hexdigest()
nbytes = sum(int(l.split(',')[1]) for l in lines)
open('/kaggle/working/file_hashes.csv', 'w').write('relative_path,size_bytes,sha256\n' + '\n'.join(lines) + '\n')
summary = dict(env=env, zip_ok=zip_ok, download_s=dl_s, unzip_s=unzip_s, hash_s=hash_s, n=len(lines), bytes=nbytes, digest=digest, digest_ok=digest == EXP_DIGEST)
print(json.dumps(summary))
json.dump(summary, open('/kaggle/working/verification_summary.json', 'w'), indent=1)
print('ALL_236_VERIFIED' if (zip_ok and len(lines) == EXP_N and nbytes == EXP_BYTES and digest == EXP_DIGEST) else 'VERIFICATION_FAILED')

In [ ]:
t0 = time.time()
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric', 'lightgbm'], capture_output=True, text=True)
import torch_geometric, lightgbm
print('pip_s', round(time.time() - t0, 1), 'rc', r.returncode, 'pyg', torch_geometric.__version__, 'lightgbm', lightgbm.__version__)